## this is to send images to gemini to get bounding box coords and plot it on image without streamlit

In [15]:
import os
import json
from PIL import Image, ImageDraw
import numpy as np
import google.generativeai as genai
from dotenv import load_dotenv
import base64
import cv2

# === Load API Key ===
load_dotenv()
model_name = "gemini-2.5-pro"
genai.configure(api_key=os.getenv("GOOGLE_GEMINI_API"))
model = genai.GenerativeModel(model_name)

# === Set the dimension value ===
dim = 1536  # Define the dimension value

# === Resize Image Function ===
def resize_image(image, dim=dim):
    image1 = np.array(image.convert('RGB'))  # Ensure the image is in RGB mode
    original_size = image1.shape  # (height, width, channels)
    image1 = image1.mean(axis=2)  # Convert image to grayscale
    h, w = image1.shape
    if w > h:
        new_w = dim
        new_h = int(h * (dim / w))
    else:
        new_h = dim
        new_w = int(w * (dim / h))
    resized_image = cv2.resize(image1, (new_w, new_h), interpolation=cv2.INTER_AREA)
    resized_image_pil = Image.fromarray(resized_image)
    resized_image_pil = resized_image_pil.convert('RGB')  # Convert to RGB before saving
    return original_size, (new_h, new_w), resized_image_pil

# === Load Image ===
def load_image(image_path):
    img = Image.open(image_path)
    return img

# === Draw Bounding Boxes on Image ===
def draw_bounding_boxes(image, coords):
    # Open image using PIL and ensure it's in RGBA mode for transparency
    image = image.convert("RGBA")
    
    # Create drawing context
    draw = ImageDraw.Draw(image)

    # Define a translucent highlight color (e.g., yellow with ~12% opacity)
    highlight_color = (190, 195, 0, 114)  # RGBA: Red, Green, Blue, Alpha (opacity: 0-255)
    
    for box in coords:
        # Ensure that the bounding box is in the form of [ymin, xmin, ymax, xmax]
        if len(box) == 4:
            ymin, xmin, ymax, xmax = [coord / 1000 for coord in box]  # Normalize to 0-1 range

            
            # Scale the coordinates to the resized image dimensions
            left = int(xmin * image.width)  # Convert to actual pixel position
            top = int(ymin * image.height)  # Convert to actual pixel position
            right = int(xmax * image.width)  # Convert to actual pixel position
            bottom = int(ymax * image.height)  # Convert to actual pixel position

            # Create a transparent overlay of the same size as the image
            overlay = Image.new('RGBA', image.size, (0, 0, 0, 0))

            # Draw the translucent background only in the bounding box region on the overlay
            overlay_draw = ImageDraw.Draw(overlay)
            overlay_draw.rectangle([left, top, right, bottom], fill=highlight_color)

            # Composite the original image with the overlay (keeping text visible)
            image = Image.alpha_composite(image, overlay)

    return image

# === Send to Gemini for OCR ===
def send_to_gemini(image, prompt):
    try:
        # Send request to Gemini with user prompt and image
        response = model.generate_content([prompt] + [image])  # Batch processing
        raw_response = response.text.strip()

        # Print raw response for debugging
        print(f"Raw response from Gemini:\n{raw_response}")

        # Clean the raw response by removing any non-JSON content (e.g., text like 'Hence')
        cleaned = raw_response.strip().lstrip("Hence").strip()  # Strip "Hence" and any leading whitespace
        cleaned = cleaned.strip('```json').strip('```').strip()  # Clean up surrounding backticks if present

        # Check if cleaned response is valid JSON
        if cleaned.startswith("[") and cleaned.endswith("]"):
            try:
                # Attempt to parse the cleaned response as JSON
                parsed = json.loads(cleaned)

                # Check the structure of the parsed result (just print it)
                print(f"Parsed result: {parsed}")

                return parsed
            except json.JSONDecodeError as e:
                print(f"❌ JSON decoding error: {e}")
                return None
        else:
            print(f"❌ Malformed JSON: {cleaned}")
            return None

    except Exception as e:
        print(f"❌ Failed to process images: {e}")
        return None

# === Example Usage ===
def process_image_with_ocr(image_path, user_prompt):
    """
    Main function to process an image with OCR and return image with bounding boxes
    """
    # Load image
    image = load_image(image_path)
    
    # Resize image
    original_size, new_size, resized_image = resize_image(image)
    
    # Send to Gemini for OCR
    results = send_to_gemini(resized_image, user_prompt)
    
    if results:
        print("OCR Results (Bounding Boxes in JSON format):")
        print(json.dumps(results, indent=2))
        
        coords = []
        
        # Handle case where the result is either a dictionary with "box_2d" or a list of coordinates
        if isinstance(results, list):
            # Check if it's a list of dictionaries with "box_2d" keys
            if all(isinstance(item, dict) and "box_2d" in item for item in results):
                # Handle case where it's a list of dictionaries like [{"box_2d": [coords], "label": "text"}]
                for item in results:
                    box_2d = item.get("box_2d", [])
                    if box_2d:
                        coords.append(box_2d)
            # If the result is a list, check if it's a list of bounding boxes or a single bounding box
            elif all(isinstance(item, list) for item in results):
                # Handle case where it's a list of bounding boxes, each item being a list like [ymin, xmin, ymax, xmax]
                coords = results
            else:
                # Handle single bounding box case, e.g., [600, 376, 626, 465]
                coords = [results]  # Wrap it in another list to make it a list of bounding boxes
        elif isinstance(results, dict):
            # Handle the case where it's a dictionary with "box_2d" key
            box_2d = results.get("box_2d", [])
            if box_2d:
                coords.append(box_2d)
        
        if coords:
            # Draw bounding boxes and return the image
            image_with_boxes = draw_bounding_boxes(resized_image, coords)
            
            # Ensure the image is in RGB mode
            image_with_boxes = image_with_boxes.convert("RGB")
            
            return image_with_boxes, results
        else:
            print("❌ No bounding boxes found in the OCR result.")
            return None, results
    else:
        print("❌ Failed to retrieve bounding boxes.")
        return None, None

# Example usage:
# image_with_boxes, ocr_results = process_image_with_ocr(
#     "path/to/your/image.jpg", 
#     "Return bounding boxes as JSON arrays on the line as [ymin, xmin, ymax, xmax]"
# )
# if image_with_boxes:
#     image_with_boxes.show()  # Display the image
#     # or save it: image_with_boxes.save("output_image.jpg")

## to break the json into images to align the json images coords and its image

In [1]:
import os
import json
import re
from pathlib import Path

def extract_page_specific_json():
    # Define paths
    images_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/labelimg/Images"
    base_trial_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial"
    output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/json"
    
    # Define all possible subject folders
    subject_folders = ["Physics", "Bio", "chemistry", "maths"]
    
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all JPEG files from images folder
    image_files = [f for f in os.listdir(images_folder) if f.endswith(('.jpeg', '.jpg'))]
    
    print(f"Found {len(image_files)} image files to process")
    
    for image_file in image_files:
        print(f"\nProcessing: {image_file}")
        
        # Extract ID and page number from filename
        match = re.match(r'(.+)_DIM_\d+_PAGE_(\d+)\.jpe?g', image_file)
        
        if not match:
            print(f"  ❌ Could not parse filename: {image_file}")
            continue
        
        file_id = match.group(1)
        page_number = int(match.group(2))
        
        print(f"  📄 ID: {file_id}")
        print(f"  📖 Page: {page_number}")
        
        # Search across all subject folders to find where this ID exists
        json_file_path = None
        found_subject = None
        
        for subject in subject_folders:
            potential_path = os.path.join(base_trial_folder, subject, "output", file_id, "json", "output_with_questions.json")
            
            if os.path.exists(potential_path):
                json_file_path = potential_path
                found_subject = subject
                print(f"  🎯 Found in {subject}: {potential_path}")
                break
        
        if not json_file_path:
            print(f"  ❌ JSON file not found in any subject folder for ID: {file_id}")
            continue
        
        try:
            # Read the JSON file
            with open(json_file_path, 'r', encoding='utf-8') as f:
                json_data = json.load(f)
            
            print(f"  📊 Loaded {len(json_data)} items from {found_subject} JSON")
            
            # Filter items where diagrams list contains a diagram with matching page_number
            filtered_data = []
            for item in json_data:
                if 'diagrams' in item and item['diagrams']:  # Check if diagrams exist and not empty
                    for diagram in item['diagrams']:
                        if 'page_number' in diagram and diagram['page_number'] == page_number:
                            filtered_data.append(item)
                            break  # Found a matching diagram, no need to check other diagrams in this item
            
            print(f"  ✅ Found {len(filtered_data)} items with diagrams on page {page_number}")
            
            if filtered_data:
                # Create output filename (same as image but with .json extension)
                output_filename = re.sub(r'\.jpe?g$', '.json', image_file)
                output_path = os.path.join(output_folder, output_filename)
                
                # Save filtered data
                with open(output_path, 'w', encoding='utf-8') as f:
                    json.dump(filtered_data, f, indent=2, ensure_ascii=False)
                
                print(f"  💾 Saved to: {output_path}")
            else:
                print(f"  ⚠️  No diagrams found for page {page_number}")
                
        except json.JSONDecodeError as e:
            print(f"  ❌ Error parsing JSON: {e}")
        except Exception as e:
            print(f"  ❌ Error processing file: {e}")
    
    print(f"\n🎉 Processing complete! Check the output folder: {output_folder}")

def list_available_images():
    """Helper function to see what images are available for processing"""
    images_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/labelimg/Images"
    
    if not os.path.exists(images_folder):
        print(f"❌ Images folder not found: {images_folder}")
        return
    
    image_files = [f for f in os.listdir(images_folder) if f.endswith(('.jpeg', '.jpg'))]
    
    print(f"📁 Found {len(image_files)} JPEG files:")
    for img in sorted(image_files):
        # Extract info from filename
        match = re.match(r'(.+)_DIM_\d+_PAGE_(\d+)\.jpe?g', img)
        if match:
            file_id = match.group(1)
            page_number = match.group(2)
            print(f"  📄 {img} → ID: {file_id}, Page: {page_number}")
        else:
            print(f"  ❓ {img} → Could not parse")

# Run the main function
if __name__ == "__main__":
    # Uncomment the line below to see available images first
    # list_available_images()
    
    # Run the extraction process
    extract_page_specific_json()

Found 57 image files to process

Processing: 07_10021157131080491171694786726_DIM_1536_PAGE_2.jpeg
  📄 ID: 07_10021157131080491171694786726
  📖 Page: 2
  🎯 Found in maths: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial/maths/output/07_10021157131080491171694786726/json/output_with_questions.json
  📊 Loaded 38 items from maths JSON
  ✅ Found 1 items with diagrams on page 2
  💾 Saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/json/07_10021157131080491171694786726_DIM_1536_PAGE_2.json

Processing: 07_100212788145098831111705212250_DIM_1536_PAGE_6.jpeg
  📄 ID: 07_100212788145098831111705212250
  📖 Page: 6
  🎯 Found in Physics: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial/Physics/output/07_100212788145098831111705212250/json/output

## this is to plot the coords in the image and save them

In [2]:
import os
import json
import numpy as np
from PIL import Image, ImageDraw
import cv2
import re

# === Load Image ===
def load_image(image_path):
    img = Image.open(image_path)
    return img

# === Draw Bounding Boxes on Image ===
def draw_bounding_boxes(image, coords):
    # Open image using PIL and ensure it's in RGBA mode for transparency
    image = image.convert("RGBA")
    
    # Create drawing context
    draw = ImageDraw.Draw(image)

    # Define a translucent highlight color (e.g., yellow with ~12% opacity)
    highlight_color = (190, 195, 0, 114)  # RGBA: Red, Green, Blue, Alpha (opacity: 0-255)
    
    for box in coords:
        # Ensure that the bounding box is in the form of [ymin, xmin, ymax, xmax]
        if len(box) == 4:
            ymin, xmin, ymax, xmax = [coord / 1000 for coord in box]  # Normalize to 0-1 range

            
            # Scale the coordinates to the resized image dimensions
            left = int(xmin * image.width)  # Convert to actual pixel position
            top = int(ymin * image.height)  # Convert to actual pixel position
            right = int(xmax * image.width)  # Convert to actual pixel position
            bottom = int(ymax * image.height)  # Convert to actual pixel position

            # Create a transparent overlay of the same size as the image
            overlay = Image.new('RGBA', image.size, (0, 0, 0, 0))

            # Draw the translucent background only in the bounding box region on the overlay
            overlay_draw = ImageDraw.Draw(overlay)
            overlay_draw.rectangle([left, top, right, bottom], fill=highlight_color)

            # Composite the original image with the overlay (keeping text visible)
            image = Image.alpha_composite(image, overlay)

    return image

def process_json_and_draw_boxes():
    # Define paths
    json_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/json"
    images_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/labelimg/Images"
    output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/images_gemini"
    
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all JSON files
    json_files = [f for f in os.listdir(json_folder) if f.endswith('.json')]
    
    print(f"Found {len(json_files)} JSON files to process")
    
    for json_file in json_files:
        print(f"\nProcessing: {json_file}")
        
        # Extract page number from JSON filename
        page_match = re.search(r'PAGE_(\d+)', json_file)
        if not page_match:
            print(f"  ❌ Could not extract page number from: {json_file}")
            continue
        
        target_page = int(page_match.group(1))
        print(f"  📖 Target page: {target_page}")
        
        # Derive corresponding image filename
        # Replace .json with .jpeg
        image_filename = json_file.replace('.json', '.jpeg')
        
        # Full paths
        json_path = os.path.join(json_folder, json_file)
        image_path = os.path.join(images_folder, image_filename)
        
        # Check if corresponding image exists
        if not os.path.exists(image_path):
            print(f"  ❌ Image not found: {image_path}")
            continue
        
        try:
            # Read JSON file
            with open(json_path, 'r', encoding='utf-8') as f:
                json_data = json.load(f)
            
            print(f"  📊 Loaded JSON with {len(json_data)} items")
            
            # Extract bounding box coordinates ONLY for diagrams matching target page
            all_coords = []
            diagram_count = 0
            skipped_count = 0
            
            for item in json_data:
                if 'diagrams' in item and item['diagrams']:
                    for diagram in item['diagrams']:
                        if 'box_2d' in diagram:
                            # Check if this diagram belongs to the target page
                            diagram_page = diagram.get('page_number')
                            if diagram_page == target_page:
                                coords = diagram['box_2d']
                                if len(coords) == 4:
                                    all_coords.append(coords)
                                    diagram_count += 1
                                    print(f"    ✅ Added diagram from page {diagram_page}: {coords}")
                            else:
                                skipped_count += 1
                                print(f"    ⏭️  Skipped diagram from page {diagram_page} (target: {target_page})")
            
            print(f"  🎯 Diagrams added: {diagram_count}")
            print(f"  ⏭️  Diagrams skipped: {skipped_count}")
            
            if all_coords:
                # Load the image directly (no resizing needed)
                original_image = load_image(image_path)
                
                print(f"  🖼️  Image size: {original_image.size}")
                
                # Draw bounding boxes only for matching page diagrams
                image_with_boxes = draw_bounding_boxes(original_image, all_coords)
                
                # Convert back to RGB for saving
                image_with_boxes = image_with_boxes.convert("RGB")
                
                # Save the image with bounding boxes
                output_filename = image_filename  # Keep the same name
                output_path = os.path.join(output_folder, output_filename)
                
                image_with_boxes.save(output_path)
                print(f"  💾 Saved image with {len(all_coords)} bounding boxes: {output_path}")
                
            else:
                print(f"  ⚠️  No diagrams found for page {target_page}")
                
        except json.JSONDecodeError as e:
            print(f"  ❌ Error parsing JSON: {e}")
        except Exception as e:
            print(f"  ❌ Error processing: {e}")
    
    print(f"\n🎉 Processing complete! Check output folder: {output_folder}")

def list_json_files_with_diagrams():
    """Helper function to see what JSON files have diagrams"""
    json_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/json"
    
    json_files = [f for f in os.listdir(json_folder) if f.endswith('.json')]
    
    print(f"📁 Analyzing {len(json_files)} JSON files:")
    
    for json_file in sorted(json_files):
        json_path = os.path.join(json_folder, json_file)
        
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                json_data = json.load(f)
            
            diagram_count = 0
            page_breakdown = {}
            
            for item in json_data:
                if 'diagrams' in item and item['diagrams']:
                    for diagram in item['diagrams']:
                        if 'box_2d' in diagram:
                            page_num = diagram.get('page_number', 'unknown')
                            page_breakdown[page_num] = page_breakdown.get(page_num, 0) + 1
                            diagram_count += 1
            
            if diagram_count > 0:
                breakdown_str = ", ".join([f"page {k}: {v}" for k, v in sorted(page_breakdown.items())])
                print(f"  📊 {json_file} → {diagram_count} diagrams ({breakdown_str})")
            else:
                print(f"  📄 {json_file} → No diagrams")
                
        except Exception as e:
            print(f"  ❌ {json_file} → Error: {e}")

# Run the main function
if __name__ == "__main__":
    # Uncomment to see which files have diagrams first
    # list_json_files_with_diagrams()
    
    # Process all JSON files and create images with bounding boxes


  process_json_and_draw_boxes()

Found 54 JSON files to process

Processing: 07_100210065532425361141704635002_DIM_1536_PAGE_3.json
  📖 Target page: 3
  📊 Loaded JSON with 1 items
    ✅ Added diagram from page 3: [753, 172, 986, 680]
  🎯 Diagrams added: 1
  ⏭️  Diagrams skipped: 0
  🖼️  Image size: (1086, 1536)
  💾 Saved image with 1 bounding boxes: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/images_gemini/07_100210065532425361141704635002_DIM_1536_PAGE_3.jpeg

Processing: 08_10021013701065351171694350774_DIM_1536_PAGE_2.json
  📖 Target page: 2
  📊 Loaded JSON with 1 items
    ✅ Added diagram from page 2: [106, 533, 201, 775]
  🎯 Diagrams added: 1
  ⏭️  Diagrams skipped: 0
  🖼️  Image size: (1086, 1536)
  💾 Saved image with 1 bounding boxes: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/images_gemini/08_1002101370106535117169

# to convert the labelimg coords to 2d to find IOU

In [3]:
import os
import json
from pathlib import Path
from PIL import Image

def get_image_dimensions(image_path):
    """Get the dimensions (width, height) of an image."""
    try:
        with Image.open(image_path) as img:
            return img.size  # returns (width, height)
    except Exception as e:
        print(f"Error reading image {image_path}: {e}")
        return None

def yolo_to_box_2d(yolo_line, image_width, image_height):
    """
    Convert YOLO format to box_2d format
    
    Args:
        yolo_line: string like "0 0.383978 0.526367 0.718232 0.453776"
        image_width: width of the image in pixels (dynamically retrieved)
        image_height: height of the image in pixels (dynamically retrieved)
    
    Returns:
        dict with class_id and box_2d coordinates
    """
    # Parse YOLO coordinates
    parts = yolo_line.strip().split()
    if len(parts) != 5:
        return None
    
    class_id = int(parts[0])
    center_x = float(parts[1])
    center_y = float(parts[2])
    width = float(parts[3])
    height = float(parts[4])
    
    # Convert to pixel coordinates
    x_min = int((center_x - width/2) * image_width)
    y_min = int((center_y - height/2) * image_height)
    x_max = int((center_x + width/2) * image_width)
    y_max = int((center_y + height/2) * image_height)
    
    return {
        "class_id": class_id,
        "box_2d": [y_min, x_min, y_max, x_max],
        "yolo_original": yolo_line.strip()
    }

def convert_yolo_txt_to_2d_json():
    # Define paths
    base_labelimg_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/labelimg"
    txt_folder = os.path.join(base_labelimg_folder, "txt")
    image_folder = os.path.join(base_labelimg_folder, "Images")  # Fixed: Capital "I" for Images
    output_folder = os.path.join(base_labelimg_folder, "2d")
    
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    print(f"📁 Created/using output folder: {output_folder}")
    
    # Check if required folders exist
    if not os.path.exists(txt_folder):
        print(f"❌ TXT folder not found: {txt_folder}")
        return
    
    if not os.path.exists(image_folder):
        print(f"❌ Images folder not found: {image_folder}")
        return
    
    # Get all .txt files
    txt_files = [f for f in os.listdir(txt_folder) if f.endswith('.txt')]
    
    if not txt_files:
        print(f"⚠️  No .txt files found in: {txt_folder}")
        return
    
    print(f"🔍 Found {len(txt_files)} .txt files to process")
    print(f"📂 Images folder: {image_folder}")
    print(f"📂 TXT folder: {txt_folder}")
    
    for txt_file in txt_files:
        print(f"\n🔄 Processing: {txt_file}")
        
        # Full path to txt file
        txt_path = os.path.join(txt_folder, txt_file)
        
        # Create corresponding JSON filename
        json_filename = txt_file.replace('.txt', '.json')
        json_path = os.path.join(output_folder, json_filename)
        
        # Construct corresponding image path based on txt file name
        # Remove .txt extension and add .jpeg
        image_filename = txt_file.replace('.txt', '.jpeg')
        image_path = os.path.join(image_folder, image_filename)
        
        print(f"  🔍 Looking for image: {image_filename}")
        
        # Check if corresponding image exists
        if not os.path.exists(image_path):
            print(f"  ❌ Image not found: {image_path}")
            print(f"     Skipping {txt_file}")
            continue
        
        # Get image dimensions dynamically
        image_dimensions = get_image_dimensions(image_path)
        if not image_dimensions:
            print(f"  ❌ Could not retrieve image dimensions for {image_path}")
            continue
        
        image_width, image_height = image_dimensions
        print(f"  📐 Image dimensions: {image_width}x{image_height}")
        
        try:
            # Read the txt file
            with open(txt_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
            
            print(f"  📄 Read {len(lines)} lines from TXT file")
            
            # Convert each line (each object) to 2D format
            converted_objects = []
            
            for line_num, line in enumerate(lines, 1):
                line = line.strip()
                if not line:  # Skip empty lines
                    continue
                
                converted = yolo_to_box_2d(line, image_width, image_height)
                if converted:
                    converted["object_id"] = line_num
                    converted_objects.append(converted)
                    print(f"    ✅ Line {line_num}: {line} → {converted['box_2d']}")
                else:
                    print(f"    ❌ Line {line_num}: Could not parse: {line}")
            
            if converted_objects:
                # Create JSON structure
                json_data = {
                    "filename": txt_file,
                    "image_filename": image_filename,
                    "image_path": image_path,
                    "image_dimensions": {
                        "width": image_width,
                        "height": image_height
                    },
                    "total_objects": len(converted_objects),
                    "objects": converted_objects
                }
                
                # Save to JSON file
                with open(json_path, 'w', encoding='utf-8') as f:
                    json.dump(json_data, f, indent=2, ensure_ascii=False)
                
                print(f"  💾 Saved {len(converted_objects)} objects to: {json_filename}")
            else:
                print(f"  ⚠️  No valid objects found in: {txt_file}")
                
        except Exception as e:
            print(f"  ❌ Error processing {txt_file}: {e}")
    
    print(f"\n🎉 Conversion complete! Check output folder: {output_folder}")

def preview_txt_and_image_pairs():
    """Helper function to preview which txt files have corresponding images"""
    base_labelimg_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/labelimg"
    txt_folder = os.path.join(base_labelimg_folder, "txt")
    image_folder = os.path.join(base_labelimg_folder, "Images")
    
    if not os.path.exists(txt_folder):
        print(f"❌ TXT folder not found: {txt_folder}")
        return
    
    if not os.path.exists(image_folder):
        print(f"❌ Images folder not found: {image_folder}")
        return
    
    txt_files = [f for f in os.listdir(txt_folder) if f.endswith('.txt')]
    image_files = [f for f in os.listdir(image_folder) if f.endswith('.jpeg')]
    
    print(f"📊 Found {len(txt_files)} TXT files and {len(image_files)} image files")
    print("\n🔍 Checking TXT-Image pairs:")
    
    for txt_file in sorted(txt_files):
        image_filename = txt_file.replace('.txt', '.jpeg')
        image_path = os.path.join(image_folder, image_filename)
        
        if os.path.exists(image_path):
            # Get image dimensions
            dimensions = get_image_dimensions(image_path)
            if dimensions:
                w, h = dimensions
                print(f"  ✅ {txt_file} → {image_filename} ({w}x{h})")
            else:
                print(f"  ⚠️  {txt_file} → {image_filename} (could not read dimensions)")
        else:
            print(f"  ❌ {txt_file} → {image_filename} (image not found)")

# Run the conversion
if __name__ == "__main__":
    # Preview txt-image pairs first
    print("🔍 PREVIEW: Checking TXT-Image pairs...")
    preview_txt_and_image_pairs()
    
    print("\n" + "="*60)
    print("🚀 STARTING CONVERSION...")
    
    # Run the conversion process
    convert_yolo_txt_to_2d_json()

🔍 PREVIEW: Checking TXT-Image pairs...
📊 Found 58 TXT files and 57 image files

🔍 Checking TXT-Image pairs:
  ✅ 01_10021165141080491171694788750_DIM_1536_PAGE_1.txt → 01_10021165141080491171694788750_DIM_1536_PAGE_1.jpeg (1086x1536)
  ✅ 01_10021165141080491171694788750_DIM_1536_PAGE_10.txt → 01_10021165141080491171694788750_DIM_1536_PAGE_10.jpeg (1086x1536)
  ✅ 01_10021165141080491171694788750_DIM_1536_PAGE_13.txt → 01_10021165141080491171694788750_DIM_1536_PAGE_13.jpeg (1536x1086)
  ✅ 01_10021165141080491171694788750_DIM_1536_PAGE_14.txt → 01_10021165141080491171694788750_DIM_1536_PAGE_14.jpeg (1086x1536)
  ✅ 01_10021165141080491171694788750_DIM_1536_PAGE_16.txt → 01_10021165141080491171694788750_DIM_1536_PAGE_16.jpeg (1086x1536)
  ✅ 01_10021165141080491171694788750_DIM_1536_PAGE_17.txt → 01_10021165141080491171694788750_DIM_1536_PAGE_17.jpeg (1536x1086)
  ✅ 01_10021165141080491171694788750_DIM_1536_PAGE_3.txt → 01_10021165141080491171694788750_DIM_1536_PAGE_3.jpeg (1086x1536)
  ✅ 01_

In [4]:
#!/usr/bin/env python3
"""
IoU Calculator for 2D Bounding Boxes
Calculates Intersection over Union between JSON files from two directories
"""

import json
import os
from typing import List, Tuple, Dict, Optional
import pandas as pd

def load_json(file_path: str) -> dict:
    """Load JSON file safely"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return {}

def extract_bbox_from_labelimg(data: dict) -> Optional[List[int]]:
    """Extract bounding box from labelimg format"""
    try:
        if 'objects' in data and len(data['objects']) > 0:
            return data['objects'][0]['box_2d']
    except (KeyError, IndexError, TypeError):
        pass
    return None

def extract_bbox_from_json(data: List[dict]) -> Optional[List[int]]:
    """Extract bounding box from diagram json format"""
    try:
        if isinstance(data, list) and len(data) > 0:
            if 'diagrams' in data[0] and len(data[0]['diagrams']) > 0:
                return data[0]['diagrams'][0]['box_2d']
    except (KeyError, IndexError, TypeError):
        pass
    return None

def calculate_iou(box1: List[int], box2: List[int]) -> float:
    """
    Calculate Intersection over Union (IoU) between two bounding boxes
    Box format: [x1, y1, x2, y2] where (x1,y1) is top-left and (x2,y2) is bottom-right
    """
    if not box1 or not box2 or len(box1) != 4 or len(box2) != 4:
        return 0.0
    
    # Extract coordinates
    x1_1, y1_1, x2_1, y2_1 = box1
    x1_2, y1_2, x2_2, y2_2 = box2
    
    # Calculate intersection coordinates
    x1_inter = max(x1_1, x1_2)
    y1_inter = max(y1_1, y1_2)
    x2_inter = min(x2_1, x2_2)
    y2_inter = min(y2_1, y2_2)
    
    # Check if there's an intersection
    if x1_inter >= x2_inter or y1_inter >= y2_inter:
        return 0.0
    
    # Calculate areas
    intersection_area = (x2_inter - x1_inter) * (y2_inter - y1_inter)
    area1 = (x2_1 - x1_1) * (y2_1 - y1_1)
    area2 = (x2_2 - x1_2) * (y2_2 - y1_2)
    union_area = area1 + area2 - intersection_area
    
    # Calculate IoU
    if union_area == 0:
        return 0.0
    
    return intersection_area / union_area

def main():
    # Directory paths
    labelimg_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/labelimg/2d"
    json_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/json"
    
    # Get all JSON files from both directories
    labelimg_files = set(f for f in os.listdir(labelimg_dir) if f.endswith('.json'))
    json_files = set(f for f in os.listdir(json_dir) if f.endswith('.json'))
    
    # Find matching files
    matching_files = labelimg_files.intersection(json_files)
    
    print(f"Found {len(matching_files)} matching files")
    print("=" * 80)
    
    results = []
    total_iou = 0
    valid_comparisons = 0
    
    for filename in sorted(matching_files):
        # Load both JSON files
        labelimg_path = os.path.join(labelimg_dir, filename)
        json_path = os.path.join(json_dir, filename)
        
        labelimg_data = load_json(labelimg_path)
        json_data = load_json(json_path)
        
        # Extract bounding boxes
        bbox1 = extract_bbox_from_labelimg(labelimg_data)
        bbox2 = extract_bbox_from_json(json_data)
        
        # Calculate IoU
        iou = calculate_iou(bbox1, bbox2)
        
        results.append({
            'filename': filename,
            'labelimg_bbox': bbox1,
            'json_bbox': bbox2,
            'iou': iou
        })
        
        if bbox1 and bbox2:
            total_iou += iou
            valid_comparisons += 1
            status = "✓" if iou > 0.5 else "⚠" if iou > 0.1 else "✗"
        else:
            status = "⚠ Missing bbox"
        
        print(f"{status} {filename}")
        print(f"   LabelImg bbox: {bbox1}")
        print(f"   JSON bbox:     {bbox2}")
        print(f"   IoU:          {iou:.4f}")
        print()
    
    # Summary statistics
    print("=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)
    print(f"Total files processed: {len(matching_files)}")
    print(f"Valid comparisons: {valid_comparisons}")
    
    if valid_comparisons > 0:
        avg_iou = total_iou / valid_comparisons
        print(f"Average IoU: {avg_iou:.4f}")
        
        # Categorize files by IoU
        high_iou_files = [r for r in results if r['labelimg_bbox'] and r['json_bbox'] and r['iou'] > 0.7]
        medium_iou_files = [r for r in results if r['labelimg_bbox'] and r['json_bbox'] and 0.3 <= r['iou'] <= 0.7]
        low_iou_files = [r for r in results if r['labelimg_bbox'] and r['json_bbox'] and r['iou'] < 0.3]
        
        # IoU distribution with filenames
        print(f"\nHigh IoU (>0.7): {len(high_iou_files)} files ({len(high_iou_files)/valid_comparisons*100:.1f}%)")
        if high_iou_files:
            for file_data in sorted(high_iou_files, key=lambda x: x['iou'], reverse=True):
                print(f"  • {file_data['filename']} (IoU: {file_data['iou']:.4f})")
        else:
            print("  (No files with high IoU)")
            
        print(f"\nMedium IoU (0.3-0.7): {len(medium_iou_files)} files ({len(medium_iou_files)/valid_comparisons*100:.1f}%)")
        if medium_iou_files:
            for file_data in sorted(medium_iou_files, key=lambda x: x['iou'], reverse=True):
                print(f"  • {file_data['filename']} (IoU: {file_data['iou']:.4f})")
        else:
            print("  (No files with medium IoU)")
            
        print(f"\nLow IoU (<0.3): {len(low_iou_files)} files ({len(low_iou_files)/valid_comparisons*100:.1f}%)")
        if low_iou_files:
            # Show top 5 and bottom 5 for brevity if many files
            if len(low_iou_files) > 10:
                print("  Top 5 with highest low IoU:")
                for file_data in sorted(low_iou_files, key=lambda x: x['iou'], reverse=True)[:5]:
                    print(f"    • {file_data['filename']} (IoU: {file_data['iou']:.4f})")
                print("  ...")
                print("  Bottom 5 with lowest IoU:")
                for file_data in sorted(low_iou_files, key=lambda x: x['iou'])[:5]:
                    print(f"    • {file_data['filename']} (IoU: {file_data['iou']:.4f})")
            else:
                for file_data in sorted(low_iou_files, key=lambda x: x['iou'], reverse=True):
                    print(f"  • {file_data['filename']} (IoU: {file_data['iou']:.4f})")
        else:
            print("  (No files with low IoU)")
    
    # Save results to CSV for further analysis
    df = pd.DataFrame(results)
    output_file = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/iou_results.csv"
    df.to_csv(output_file, index=False)
    print(f"\nDetailed results saved to: {output_file}")

if __name__ == "__main__":
    main()

Found 54 matching files
✗ 01_10021165141080491171694788750_DIM_1536_PAGE_1.json
   LabelImg bbox: [1058, 106, 1279, 485]
   JSON bbox:     [679, 99, 868, 361]
   IoU:          0.0000

✗ 01_10021165141080491171694788750_DIM_1536_PAGE_10.json
   LabelImg bbox: [528, 103, 752, 496]
   JSON bbox:     [234, 173, 519, 429]
   IoU:          0.0000

⚠ 01_10021165141080491171694788750_DIM_1536_PAGE_13.json
   LabelImg bbox: [139, 96, 373, 438]
   JSON bbox:     [140, 136, 311, 361]
   IoU:          0.4808

⚠ 01_10021165141080491171694788750_DIM_1536_PAGE_14.json
   LabelImg bbox: [459, 27, 1156, 807]
   JSON bbox:     [112, 106, 893, 744]
   IoU:          0.3619

✗ 01_10021165141080491171694788750_DIM_1536_PAGE_16.json
   LabelImg bbox: [941, 230, 1201, 559]
   JSON bbox:     [387, 119, 613, 407]
   IoU:          0.0000

✗ 01_10021165141080491171694788750_DIM_1536_PAGE_17.json
   LabelImg bbox: [667, 852, 828, 1084]
   JSON bbox:     [666, 549, 829, 868]
   IoU:          0.0297

⚠ 01_1002116514

In [5]:
import json
import os
from typing import List

def load_json(file_path: str) -> dict:
    """Load JSON file safely"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return {}

def calculate_iou(box1: List[int], box2: List[int]) -> float:
    """
    Calculate Intersection over Union (IoU) between two bounding boxes
    Box format: [x1, y1, x2, y2] where (x1,y1) is top-left and (x2,y2) is bottom-right
    """
    if not box1 or not box2 or len(box1) != 4 or len(box2) != 4:
        return 0.0
    
    # Extract coordinates
    x1_1, y1_1, x2_1, y2_1 = box1
    x1_2, y1_2, x2_2, y2_2 = box2
    
    # Calculate intersection coordinates
    x1_inter = max(x1_1, x1_2)
    y1_inter = max(y1_1, y1_2)
    x2_inter = min(x2_1, x2_2)
    y2_inter = min(y2_1, y2_2)
    
    # Check if there's an intersection
    if x1_inter >= x2_inter or y1_inter >= y2_inter:
        return 0.0
    
    # Calculate areas
    intersection_area = (x2_inter - x1_inter) * (y2_inter - y1_inter)
    area1 = (x2_1 - x1_1) * (y2_1 - y1_1)
    area2 = (x2_2 - x1_2) * (y2_2 - y1_2)
    union_area = area1 + area2 - intersection_area
    
    # Calculate IoU
    if union_area == 0:
        return 0.0
    
    return intersection_area / union_area

def extract_bbox_from_labelimg(data: dict) -> List[int]:
    """Extract bounding box from labelimg format"""
    try:
        if 'objects' in data and len(data['objects']) > 0:
            return data['objects'][0]['box_2d']
    except (KeyError, IndexError, TypeError):
        pass
    return []

def extract_bbox_from_json(data: List[dict]) -> List[int]:
    """Extract bounding box from diagram json format"""
    try:
        if isinstance(data, list) and len(data) > 0:
            if 'diagrams' in data[0] and len(data[0]['diagrams']) > 0:
                return data[0]['diagrams'][0]['box_2d']
    except (KeyError, IndexError, TypeError):
        pass
    return []

def main():
    # Directory paths
    json_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/json"
    labelimg_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z3_ocr_gemini_pcmb_13.3/diagram_detection/labelimg/2d"
    
    # Get all JSON files from both directories
    labelimg_files = set(f for f in os.listdir(labelimg_dir) if f.endswith('.json'))
    json_files = set(f for f in os.listdir(json_dir) if f.endswith('.json'))
    
    # Find matching files
    matching_files = labelimg_files.intersection(json_files)
    
    print(f"Found {len(matching_files)} matching files")
    print("=" * 80)
    
    results = []
    total_iou = 0
    valid_comparisons = 0
    
    # Iterate over matching files
    for filename in sorted(matching_files):
        # Load both JSON files
        labelimg_path = os.path.join(labelimg_dir, filename)
        json_path = os.path.join(json_dir, filename)

        labelimg_data = load_json(labelimg_path)
        json_data = load_json(json_path)

        # Extract bounding boxes
        bbox1 = extract_bbox_from_labelimg(labelimg_data)
        bbox2 = extract_bbox_from_json(json_data)

        # Calculate IoU
        iou = calculate_iou(bbox1, bbox2)

        # Print each IoU value
        print(f"{filename}: IoU = {iou:.4f}")
        
        # Accumulate IoU and count valid comparisons
        if bbox1 and bbox2:
            total_iou += iou
            valid_comparisons += 1

    # Calculate average IoU
    if valid_comparisons > 0:
        avg_iou = total_iou / valid_comparisons
    else:
        avg_iou = 0.0
    
    print("=" * 80)
    print(f"Average IoU: {avg_iou:.4f}")
    print("=" * 80)
    
    # Optionally, save results to CSV or further process them
    # For now, we're printing out the average IoU directly.
    
if __name__ == "__main__":
    main()


Found 54 matching files
01_10021165141080491171694788750_DIM_1536_PAGE_1.json: IoU = 0.0000
01_10021165141080491171694788750_DIM_1536_PAGE_10.json: IoU = 0.0000
01_10021165141080491171694788750_DIM_1536_PAGE_13.json: IoU = 0.4808
01_10021165141080491171694788750_DIM_1536_PAGE_14.json: IoU = 0.3619
01_10021165141080491171694788750_DIM_1536_PAGE_16.json: IoU = 0.0000
01_10021165141080491171694788750_DIM_1536_PAGE_17.json: IoU = 0.0297
01_10021165141080491171694788750_DIM_1536_PAGE_3.json: IoU = 0.4730
01_10021165141080491171694788750_DIM_1536_PAGE_8.json: IoU = 0.0000
02_10021028581065351171694350945_DIM_1536_PAGE_1.json: IoU = 0.0000
02_10021028581065351171694350945_DIM_1536_PAGE_18.json: IoU = 0.5689
02_10021028581065351171694350945_DIM_1536_PAGE_20.json: IoU = 0.0402
02_10021028581065351171694350945_DIM_1536_PAGE_5.json: IoU = 0.0901
02_10021028581065351171694350945_DIM_1536_PAGE_9.json: IoU = 0.0000
04_10021039411060911141694339166_DIM_1536_PAGE_2.json: IoU = 0.0000
04_10021039411060